# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# .to_json() provides a deep dict snapshot of all metadata fields
metadata = dataset.metadata.to_json()

print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, their IDs, and the available fields (columns) for each record set.

We utilize the model structure to list details; for each record set, we show its `@id`, label, and the list of fields and field @ids.

In [ ]:
# List available record sets and their fields by their @id
print("Available Record Sets (by @id):")
record_sets_md = dataset.metadata.record_sets
for rs in record_sets_md:
    print(f"- Record Set @id: {rs['@id']}")
    if 'name' in rs:
        print(f"  Name: {rs['name']}")
    if 'description' in rs:
        print(f"  Description: {rs['description']}")
    if 'field' in rs:
        print("  Fields:")
        for field in rs['field']:
            print(f"    - Field @id: {field['@id']} (label: {field.get('name', '')})")
    print()

print("First sample records from each record set:")
for rs in record_sets_md:
    rs_id = rs['@id']
    print(f"Sample from Record Set {rs_id}:")
    for i, x in enumerate(dataset.records(record_set=rs_id)):
        print(x)
        if i >= 1:
            break
    print()

## 3. Data Extraction
Load data from specific record set(s) into DataFrame(s) for analysis. We will use the record set and field `@id`s as discovered above.

**Tip**: Replace the `record_sets` variable content with the actual record set `@id`s from previous output if you want all record sets.

In [ ]:
# Extract data from each record set into Pandas DataFrames
# Get all record set @ids dynamically
record_sets = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

# For demonstration, show the columns of the main survey record set
# We'll try to auto-detect one with the word "survey" or the most populated
main_rs_id = None
max_cols = -1
for k, df in dataframes.items():
    print(f"Record set {k} columns: {df.columns.tolist()}")
    if df.shape[1] > max_cols:
        max_cols = df.shape[1]
        main_rs_id = k

print(f"\nUsing main record set: {main_rs_id}")
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps—filter records, normalize numeric fields, and group by relevant key attributes.
All operations reference columns by their field `@id`.

In [ ]:
# Choose numeric field and group field by inspecting the columns of the main record set
df = dataframes[main_rs_id]
print(f"Available columns (field @id): {df.columns.tolist()}")
# Example numeric field: 'PHQ-9_score' or 'GAD-7_score' @id, let's try to infer it

import re
numeric_field = None
for col in df.columns:
    if re.search(r'phq|gad|psq', col, re.I) and re.search(r'score', col, re.I):
        numeric_field = col
        print(f"Using numeric field: {numeric_field}")
        break
if not numeric_field:
    numeric_field = df.select_dtypes('number').columns[0] # fallback
    print(f"Fallback numeric field: {numeric_field}")

# Group field: try demographic, eg. 'cr:field:gender' or 'cr:field:age_group'.
group_field = None
for col in df.columns:
    if re.search(r'gender|age|marital|education', col, re.I):
        group_field = col
        print(f"Using group field: {group_field}")
        break
if not group_field:
    group_field = df.columns[0]
    print(f"Fallback group field: {group_field}")

# Filter records: show those where numeric_score is above threshold (mean for demonstration)
threshold = df[numeric_field].mean()
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.1f}:")
print(filtered_df[[numeric_field, group_field]].head())

# Normalize numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group and aggregate
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"\nGrouped data by {group_field}, mean {numeric_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the chosen numeric field and its grouping by the selected group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field], kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Boxplot by group field
if group_field is not None and group_field in df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
We successfully loaded and explored the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using `mlcroissant`. By referencing all entities by their `@id`, we:
- Inspected record sets, fields, and their structure
- Loaded records as pandas DataFrames for flexible analysis
- Filtered and normalized a core survey score field (e.g., PHQ-9 or GAD-7 score)
- Conducted grouping by demographic field (e.g., gender, age)
- Visualized score distributions and demographic differences

This workflow demonstrates how to use Croissant-standardized, AI-ready tabular data for robust, reproducible data science processes.